In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.olist_landing

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.olist_landing.raw_files

In [0]:
orders_df = spark.read.csv("/Volumes/workspace/olist_landing/raw_files/olist_orders_dataset.csv" ,
                           header=True,
                           inferSchema=True)

display(orders_df)
orders_df.printSchema()


In [0]:
from pyspark.sql import Window # difined window function
from pyspark.sql.functions import ntile, count # Ntile: Divide the rows into buckets based on the value of an expression

if orders_df.filter("order_purchase_timestamp IS NULL").count() > 0:
    raise ValueError("The are oders without purchase timestamp")
    
batch_window = Window.orderBy("order_purchase_timestamp", "order_id")
orders_batched_df = orders_df.withColumn("batch_id", ntile(3).over(batch_window))

display(orders_batched_df
        .groupBy("batch_id")
        .count()
        .orderBy("batch_id")
        )




In [0]:
itens_df = spark.read.csv("/Volumes/workspace/olist_landing/raw_files/olist_order_items_dataset.csv" ,
                           header=True,
                           inferSchema=True)

orders_batched_df = orders_batched_df.select("order_id", "batch_id")
item_batched_df = itens_df.join(orders_batched_df, 
                                on="order_id", 
                                how = "left")
display(item_batched_df.groupBy("batch_id").count().orderBy("batch_id"))


In [0]:
order_payment_df = spark.read.csv("/Volumes/workspace/olist_landing/raw_files/olist_order_payments_dataset.csv" ,
                           header=True,
                           inferSchema=True)

orders_batched_df = orders_batched_df.select("order_id", "batch_id")
order_payment_batched_df = order_payment_df.join(orders_batched_df, 
                                on="order_id", 
                                how = "left")
display(order_payment_batched_df.groupBy("batch_id").count().orderBy("batch_id"))

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.olist_landing.batched_files


In [0]:
base_path = "/Volumes/workspace/olist_landing/batched_files/"

# dicionary to store the dataframes
data_set = {
    "orders": orders_batched_df,
    "items": item_batched_df,
    "payments": order_payment_batched_df
}



In [0]:
batch_id = 1
for name, df in data_set.items():
    batch_df = df.filter(df.batch_id == batch_id).drop("batch_id")
    output_path = f"{base_path}/batch_{batch_id}/{name}"
    (
        batch_df.write
        .mode("error")
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .csv(output_path)
    )


In [0]:
batch_id = 2
for name, df in data_set.items():
    batch_df = df.filter(df.batch_id == batch_id).drop("batch_id")
    output_path = f"{base_path}/batch_{batch_id}/{name}"
    (
        batch_df.write
        .mode("error")
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .csv(output_path)
    )


In [0]:
batch_id = 3
for name, df in data_set.items():
    batch_df = df.filter(df.batch_id == batch_id).drop("batch_id")
    output_path = f"{base_path}/batch_{batch_id}/{name}"
    (
        batch_df.write
        .mode("error")
        .option("header", True)
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .csv(output_path)
    )